# ECG Classification - LSTM Model (v3)

This notebook implements an LSTM-based approach for ECG binary classification.

**Input**: 188 columns of ECG time series data
**Output**: Binary classification (0 = normal, 1 = abnormal)

**Approach**: LSTM networks are well-suited for sequential data like ECG signals as they can capture temporal dependencies in the heartbeat patterns.

In [ ]:
# Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Bidirectional, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
import warnings
warnings.filterwarnings('ignore')

## Data Loading

In [ ]:
# Load data - adjust path as needed
try:
    df1 = pd.read_csv('/kaggle/input/ecg-dataset/ecg.csv', header=None)
    df2 = pd.read_csv('/kaggle/input/ecg2-dataset/ecg3.csv', header=None)
    df2 = df2.rename(columns={0: 'orig_0'})
    df2.insert(0, 0, df2['orig_0'])
    df2.columns = range(df2.shape[1])
    df = pd.concat([df1, df2], ignore_index=True)
except:
    df = pd.read_csv('../../dataset_aritmia_NEW.csv')

print(f'Dataset shape: {df.shape}')

In [ ]:
# Add column names
n_features = df.shape[1] - 1
column_names = [f'f{i}' for i in range(n_features)] + ['label']
df.columns = column_names
print(f'Number of features: {n_features}')
df.head()

In [ ]:
# Check label distribution
print('Label distribution:')
print(df['label'].value_counts())
print('\nPercentage:')
print(df['label'].value_counts(normalize=True) * 100)

## Data Preprocessing

In [ ]:
# Prepare features and labels
X = df.drop('label', axis=1).values
y = df['label'].values

# Normalize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Reshape for LSTM: (samples, timesteps, features)
X_scaled = X_scaled.reshape((X_scaled.shape[0], X_scaled.shape[1], 1))

# Encode labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)
y_categorical = to_categorical(y_encoded)

print(f'X shape: {X_scaled.shape}')
print(f'y shape: {y_categorical.shape}')
print(f'Classes: {le.classes_}')

In [ ]:
# Train-test split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_categorical, test_size=0.2, random_state=42, stratify=y_encoded
)

# Compute class weights
y_train_classes = np.argmax(y_train, axis=1)
class_weights = compute_class_weight('balanced', classes=np.unique(y_train_classes), y=y_train_classes)
class_weight_dict = dict(enumerate(class_weights))
print(f'Class weights: {class_weight_dict}')
print(f'\nTrain shape: {X_train.shape}')
print(f'Test shape: {X_test.shape}')

## Bidirectional LSTM Model Architecture

In [ ]:
def create_lstm_model(input_shape, num_classes):
    model = Sequential([
        # Bidirectional LSTM layer 1
        Bidirectional(LSTM(64, return_sequences=True), input_shape=input_shape),
        BatchNormalization(),
        Dropout(0.3),
        
        # Bidirectional LSTM layer 2
        Bidirectional(LSTM(32, return_sequences=False)),
        BatchNormalization(),
        Dropout(0.3),
        
        # Dense layers
        Dense(64, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dropout(0.2),
        Dense(num_classes, activation='softmax')
    ])
    return model

model = create_lstm_model(
    input_shape=(X_train.shape[1], 1),
    num_classes=y_categorical.shape[1]
)

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# Callbacks
callbacks = [
    ModelCheckpoint(
        'ecg_lstm_model.h5',
        monitor='val_loss',
        save_best_only=True,
        mode='min',
        verbose=1
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=25,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=8,
        min_lr=1e-6,
        verbose=1
    )
]

In [ ]:
# Train the model
history = model.fit(
    X_train, y_train,
    epochs=150,
    batch_size=32,
    validation_data=(X_test, y_test),
    callbacks=callbacks,
    class_weight=class_weight_dict,
    verbose=1
)

## Training Visualization

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'], label='Train Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy')
axes[0].set_title('LSTM Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history.history['loss'], label='Train Loss')
axes[1].plot(history.history['val_loss'], label='Validation Loss')
axes[1].set_title('LSTM Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## Model Evaluation

In [ ]:
# Evaluate model
loss, accuracy = model.evaluate(X_test, y_test)
print(f'\nTest Loss: {loss:.4f}')
print(f'Test Accuracy: {accuracy:.4f}')

In [ ]:
# Predictions and Confusion Matrix
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true_classes = np.argmax(y_test, axis=1)

cm = confusion_matrix(y_true_classes, y_pred_classes)
plt.figure(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
disp.plot(cmap='Blues')
plt.title('Confusion Matrix - LSTM Model')
plt.show()

In [ ]:
# Classification Report
target_names = [str(label) for label in le.classes_]
print('\nClassification Report:')
print(classification_report(y_true_classes, y_pred_classes, target_names=target_names))

In [ ]:
# ROC-AUC Score
roc_auc = roc_auc_score(y_test, y_pred, multi_class='ovr')
print(f'ROC-AUC Score: {roc_auc:.4f}')

## Prediction Function

In [ ]:
def predict_ecg_lstm(signal, model, scaler, label_encoder):
    signal = np.array(signal).reshape(1, -1)
    signal_scaled = scaler.transform(signal)
    signal_scaled = signal_scaled.reshape(1, -1, 1)
    y_pred = model.predict(signal_scaled, verbose=0)
    predicted_class = np.argmax(y_pred)
    confidence = np.max(y_pred)
    predicted_label = label_encoder.inverse_transform([predicted_class])[0]
    return predicted_label, confidence

In [ ]:
# Test prediction with sample data (baseline ~958, peaks during beats)
sample_data = [
    952, 954, 956, 955, 955, 953, 952, 952, 951, 955, 953, 954, 952, 953, 952, 955,
    957, 958, 958, 962, 963, 964, 963, 965, 963, 967, 969, 971, 973, 973, 972, 971,
    973, 973, 972, 968, 966, 968, 970, 973, 969, 966, 960, 964, 965, 970, 971, 970,
    967, 964, 962, 961, 960, 954, 950, 951, 951, 951, 950, 950, 950, 948, 952, 949,
    949, 944, 940, 943, 944, 947, 948, 944, 941, 943, 945, 945, 945, 942, 938, 936,
    933, 926, 927, 920, 910, 909, 919, 940, 963, 986, 1020, 1068, 1121, 1167, 1193,
    1201, 1182, 1136, 1069, 1010, 967, 939, 924, 914, 917, 924, 931, 934, 934, 937,
    940, 942, 942, 941, 940, 939, 940, 938, 938, 936, 934, 934, 938, 938, 938, 935,
    935, 936, 939, 938, 938, 939, 935, 935, 938, 937, 937, 935, 935, 937, 938, 939,
    938, 938, 936, 940, 938, 939, 938, 936, 936, 938, 938, 941, 939, 938, 933, 935,
    935, 938, 938, 936, 936, 937, 938, 941, 941, 939, 939, 940, 943, 943, 941, 941,
    939, 938, 943, 943, 943, 943, 938, 941, 942, 941, 938, 935, 931, 932
]

predicted_label, confidence = predict_ecg_lstm(sample_data, model, scaler, le)
print(f'Prediction: {predicted_label}')
print(f'Confidence: {confidence:.4f}')
print(f'Expected: Normal (0)')

## Save Model

In [ ]:
# Save the final model
model.save('ecg_lstm_final.h5')
print('LSTM Model saved successfully!')